# Arabic Vocabulary Reduction Analysis

This notebook analyzes Arabic tokens in model tokenizers and compares the size of dotted vs undotted (Rasm) token vocabularies to measure the potential vocabulary reduction from using undotted Arabic text.

In [1]:
from transformers import AutoTokenizer
from tnqeet import remove_dots
from tnqeet.constants import ARABIC_LETTERS

In [2]:
def contains_arabic(text: str) -> bool:
    """Check if a string contains any Arabic letters."""
    return any(char in ARABIC_LETTERS for char in text)


def analyze_tokenizer_arabic_vocab(model_path: str) -> dict:
    """
    Analyze a tokenizer's vocabulary for Arabic tokens.
    
    Returns a dict with:
    - model_path: the model path
    - total_vocab_size: total vocabulary size
    - arabic_tokens_count: number of tokens containing Arabic letters
    - dotless_tokens_count: number of unique tokens after undotting
    - reduction_ratio: percentage reduction from dotted to dotless
    """
    print(f"Loading tokenizer from: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    
    vocab = tokenizer.get_vocab()
    print(f"Total vocabulary size: {len(vocab)}")
    
    # Collect tokens containing Arabic letters (list since tokenizer has unique tokens)
    arabic_tokens = set()
    for token in vocab.keys():
        # Decode the token to get the actual text
        decoded_token = tokenizer.convert_tokens_to_string([token]).strip()
        if contains_arabic(decoded_token):
            arabic_tokens.add(decoded_token)
    
    print(f"Arabic tokens set size: {len(arabic_tokens)}")
    
    # Undot the Arabic tokens and add to a set
    dotless_tokens = set()
    for token in arabic_tokens:
        dotless_token = remove_dots(token)
        dotless_tokens.add(dotless_token)
    
    print(f"Dotless tokens set size: {len(dotless_tokens)}")
    
    # Calculate reduction ratio
    if len(arabic_tokens) > 0:
        reduction_ratio = (len(arabic_tokens) - len(dotless_tokens)) / len(arabic_tokens) * 100
    else:
        reduction_ratio = 0.0
    
    print(f"Reduction ratio: {reduction_ratio:.2f}%")
    print("-" * 50)
    
    return {
        "model_path": model_path,
        "total_vocab_size": len(vocab),
        "arabic_tokens_count": len(arabic_tokens),
        "dotless_tokens_count": len(dotless_tokens),
        "reduction_ratio": reduction_ratio,
    }

In [3]:
# List of models to analyze
models = [
    "google/gemma-3-1b-it",
    'openai/gpt-oss-120b',
    "Qwen/Qwen2.5-3B-Instruct",
    "meta-llama/Llama-3.2-1B-Instruct",
    'aubmindlab/bert-base-arabert',
    'humain-ai/ALLaM-7B-Instruct-preview',
    'FreedomIntelligence/AceGPT-v2-8B-Chat',
    'inceptionai/Jais-2-8B-Chat',
    'CAMeL-Lab/bert-base-arabic-camelbert-msa',
    'UBC-NLP/MARBERT',
    'UBC-NLP/AraT5-base',
    'lanwuwei/GigaBERT-v3-Arabic-and-English',
    'QCRI/Fanar-1-9B-Instruct',
    'CohereLabs/c4ai-command-a-03-2025',
    'microsoft/Phi-4-mini-instruct',
    # 'Navid-AI/Yehia-7B-preview',
    
]

In [4]:
# Analyze all models
results = []
for model_path in models:
    result = analyze_tokenizer_arabic_vocab(model_path)
    results.append(result)

Loading tokenizer from: google/gemma-3-1b-it
Total vocabulary size: 262145
Arabic tokens set size: 6938
Dotless tokens set size: 5925
Reduction ratio: 14.60%
--------------------------------------------------
Loading tokenizer from: openai/gpt-oss-120b
Total vocabulary size: 200019
Arabic tokens set size: 6835
Dotless tokens set size: 5820
Reduction ratio: 14.85%
--------------------------------------------------
Loading tokenizer from: Qwen/Qwen2.5-3B-Instruct
Total vocabulary size: 151665
Arabic tokens set size: 3272
Dotless tokens set size: 2806
Reduction ratio: 14.24%
--------------------------------------------------
Loading tokenizer from: meta-llama/Llama-3.2-1B-Instruct
Total vocabulary size: 128256
Arabic tokens set size: 3135
Dotless tokens set size: 2650
Reduction ratio: 15.47%
--------------------------------------------------
Loading tokenizer from: aubmindlab/bert-base-arabert
Total vocabulary size: 64000
Arabic tokens set size: 52057
Dotless tokens set size: 41092
Reduct

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Total vocabulary size: 110000
Arabic tokens set size: 58824
Dotless tokens set size: 48009
Reduction ratio: 18.39%
--------------------------------------------------
Loading tokenizer from: lanwuwei/GigaBERT-v3-Arabic-and-English
Total vocabulary size: 50000
Arabic tokens set size: 26168
Dotless tokens set size: 23217
Reduction ratio: 11.28%
--------------------------------------------------
Loading tokenizer from: QCRI/Fanar-1-9B-Instruct
Total vocabulary size: 128256
Arabic tokens set size: 3788
Dotless tokens set size: 3095
Reduction ratio: 18.29%
--------------------------------------------------
Loading tokenizer from: CohereLabs/c4ai-command-a-03-2025
Total vocabulary size: 255033
Arabic tokens set size: 5610
Dotless tokens set size: 4856
Reduction ratio: 13.44%
--------------------------------------------------
Loading tokenizer from: microsoft/Phi-4-mini-instruct
Total vocabulary size: 200029
Arabic tokens set size: 6835
Dotless tokens set size: 5820
Reduction ratio: 14.85%
---

In [5]:
# Summary table
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"{'Model':<50} {'Arabic':<10} {'Dotless':<10} {'Reduction':<10}")
print("-" * 80)
for r in sorted(results, key=lambda x: x['reduction_ratio'], reverse=True):
    model_name = r['model_path'].split('/')[-1]
    print(f"{model_name:<50} {r['arabic_tokens_count']:<10} {r['dotless_tokens_count']:<10} {r['reduction_ratio']:.2f}%")


SUMMARY
Model                                              Arabic     Dotless    Reduction 
--------------------------------------------------------------------------------
bert-base-arabert                                  52057      41092      21.06%
bert-base-arabic-camelbert-msa                     26798      21542      19.61%
ALLaM-7B-Instruct-preview                          34098      27435      19.54%
AraT5-base                                         58824      48009      18.39%
Fanar-1-9B-Instruct                                3788       3095       18.29%
MARBERT                                            90877      76048      16.32%
Jais-2-8B-Chat                                     43128      36186      16.10%
Llama-3.2-1B-Instruct                              3135       2650       15.47%
AceGPT-v2-8B-Chat                                  3135       2650       15.47%
gpt-oss-120b                                       6835       5820       14.85%
Phi-4-mini-instruct       

In [6]:
sum(r['reduction_ratio'] for r in results) / len(results)

16.23418474893902